In [ ]:
import pygmt
import xarray as xr
import rioxarray as rxr 
import pandas as pd
import numpy as np

fig = pygmt.Figure()
fig.coast(
    projection="G14.5/36/15c+a12+t45+v60/60+w0+z200",
    region="g",
    frame=["x10g10", "y5g5"],
    land="gray",
)


# Load the grid
grid_CT = xr.open_dataset('/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/data/processed/CT_defbathy.nc',engine='netcdf4')
grid_SR = xr.open_dataset('/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/data/processed/SR_defbathy.nc',engine='netcdf4')

#plot CT and SR grids area as rectangles
fig.plot(
    data = np.array([[float(grid_CT['x'].min()),
                    float(grid_CT['y'].min()),
                    float(grid_CT['x'].max()),
                    float(grid_CT['y'].max())]]),
    style='r+s',
    pen= "1p,black,-",)

fig.plot(
    data = np.array([[float(grid_SR['x'].min()),
                    float(grid_SR['y'].min()),
                    float(grid_SR['x'].max()),
                    float(grid_SR['y'].max())]]),
    style='r+s',
    pen= "1p,black,-",)

fig.show()



In [ ]:
import pygmt
import xarray as xr
import rioxarray as rxr 
import pandas as pd
import numpy as np


# Load the grid
grid_CT = xr.open_dataset('/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/data/processed/CT_defbathy.nc',engine='netcdf4')
grid_SR = xr.open_dataset('/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/data/processed/SR_defbathy.nc',engine='netcdf4')

grid = pygmt.datasets.load_earth_relief(
    resolution="01s", region=[14.75, 15.6,36.7, 37.7], registration="gridline"
)
# # #hillshade
# dgrid = pygmt.grdgradient(grid=grid_bathy, radiance=[270, 30])
# hgrid = pygmt.grdgradient(grid=grid_topo, radiance=[270, 30])

#read offshore points from 
gauges = pd.read_csv('/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/resources/raw/pois_depth.csv',sep=',')

CT_gauges = list(range(35,44))
SR_gagues = list(range(53,58))

#combine both
selected_gauges =  gauges.iloc[CT_gauges + SR_gagues]

fig = pygmt.Figure()
pygmt.makecpt(cmap="cmocean/topo", series=[-4000, 500])
fig.grdimage(grid=grid, frame="a", projection="M15c", cmap=True,shading=True)
fig.grdcontour(grid=grid, interval=250, annotation=500,limit=[-3500, -10])
fig.grdcontour(grid=grid, interval=100, annotation=100,limit=[-101, -99])
fig.grdcontour(grid=grid, interval=50,limit=[-51, -49],pen="0.75p,Grey36,--",label_placement="D12m",annotation='50')
#plot CT and SR grids area as rectangles
fig.plot(
    data = np.array([[float(grid_CT['x'].min()),
                    float(grid_CT['y'].min()),
                    float(grid_CT['x'].max()),
                    float(grid_CT['y'].max())]]),
    style='r+s',
    pen= "1p,black,-",)

fig.plot(
    data = np.array([[float(grid_SR['x'].min()),
                    float(grid_SR['y'].min()),
                    float(grid_SR['x'].max()),
                    float(grid_SR['y'].max())]]),
    style='r+s',
    pen= "1p,black,-",)

fig.plot(
    x=selected_gauges['lon'],
    y=selected_gauges['lat'],
    style="c0.15+r",
    pen="2p,red",
    label="Gauges ",
)

#add annotations for gauges
for i in range(len(selected_gauges)):
    fig.text(
        x=selected_gauges['lon'].iloc[i],
        y=selected_gauges['lat'].iloc[i],
        text=selected_gauges['id'].iloc[i],
        font="11p,Helvetica-Bold,black",
        justify="CM",
        offset="0.35c",
    )

fig.text(
    x=15.07,    
    y=37.29,
    text="Catania site",
    font="15p,Helvetica-Bold,black",
    justify="CM",
    offset="0.1c",
)

fig.text(
    x=15.18,    
    y=36.98,
    text="Siracusa site",
    font="15p,Helvetica-Bold,black",
    justify="CM",
    offset="0.1c",
)

# fig.colorbar(frame=["a500", "x+lElevation", "y+lm"])
fig.legend()
fig.savefig('./model_regions.png',dpi=300)
fig.show()


In [ ]:
import pygmt
import xarray as xr
import rioxarray as rxr 
import pandas as pd
import numpy as np

# Load the grid
grid = pygmt.datasets.load_earth_relief(
    resolution="03s", region=[12,20,35,39.5], registration="gridline"
)
#read offshore points from 
gauges = pd.read_csv('../../../resources/raw/pois_depth.csv',sep=',')[5:71]
fig = pygmt.Figure()
with pygmt.config(MAP_FRAME_TYPE="fancy+",FORMAT_GEO_MAP="ddd.xx"):
    pygmt.makecpt(cmap="cmocean/topo", series=[-4500, 500,10],continuous=False)
    fig.grdimage(grid=grid, frame="a", projection="M25c", cmap=True,shading=True)
    #plot region 14.75, 15.6,36.7, 37.7 area as rectangles
    fig.plot(data = np.array([[14.75, 36.7,
                15.6, 37.7]]),
                style='r+s',
                pen= "1p,black,--",)
    fig.plot(
        x=gauges['lon'],
        y=gauges['lat'],
        style="p0.03c", 
        fill="red",
        label="Gauges ",
    )
    fig.colorbar(frame=["a500", "x+lElevation", "y+lm"])
    fig.savefig('./model_regions_large.png',dpi=300)
fig.show()


Performance Map

In [ ]:
import pygmt
import pandas as pd
import matplotlib.pyplot as plt
import contextily as cx
import numpy as np

# Load file
MLDir = '/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami' 
reg = 'CT'
if reg == 'CT':
    columnname = str(38)
    list_size = ['225', '529','892','1658','3454','7071']  
elif reg == 'SR':
    columnname = str(54)
    list_size = ['228', '550' ,'961','1773','3669','6941']

train_size = list_size[3]
eve_perf = pd.read_csv(f'{MLDir}/model/{reg}/multifoldMC/out/model_direct_off[64, 128, 256]_on[16, 128, 128]_{train_size}_compile_combined.csv')
eve_perf = eve_perf.sort_values(by='g', ascending=True)
eve_perf.reset_index(drop=True, inplace=True)

def filter_grids(samplelist):
    empty_table = pd.DataFrame()
    #filter rows per each grid_id, and keep row with lowest g value
    print('count',len(samplelist))
    for grid in samplelist['Location'].unique():
        samplegrid = samplelist[samplelist['Location']==grid]
        samplegrid = samplegrid.sort_values(by='g', ascending=False)
        #remove all rows except first one
        samplegrid = samplegrid.drop(samplegrid.index[1:])
        #append to empty table
        empty_table = pd.concat([empty_table,samplegrid],axis=0)
    return empty_table, len(samplelist)    

sample_train = eve_perf[eve_perf['split']=='train']
sample_train = sample_train[sample_train['max_off']>0.1]
# sample_train = sample_train[sample_train['g']>0.3]
sample_train = sample_train[sample_train['true']>5000]
sample_train,train_count = filter_grids(sample_train)

sample_test = eve_perf[eve_perf['split']=='test']
sample_test = sample_test[sample_test['max_off']>0.1]
sample_test,test_count = filter_grids(sample_test)

sample_bad = eve_perf[eve_perf['split']=='test']
sample_bad = sample_bad[sample_bad['max_off']>0.1]
sample_bad = sample_bad[sample_bad['g']>0.3]
sample_bad,bad_count = filter_grids(sample_bad)
# bad_count = len(sample_bad)

sample_bad_bigger = eve_perf[eve_perf['split']=='test']
sample_bad_bigger = sample_bad_bigger[sample_bad_bigger['max_off']>0.1]
# sample_bad_bigger = sample_bad_bigger[sample_bad_bigger['g']>0.3]
sample_bad_bigger= sample_bad_bigger[sample_bad_bigger['true']>5000]
sample_bad_bigger,realbad_count = filter_grids(sample_bad_bigger)

# Set region and projection
region = [12, 28, 32, 41]  # same as xlim/ylim
projection = "M15c"  # Mercator projection, 15 cm width

# Create two subplot panels
fig = pygmt.Figure()
grid = pygmt.datasets.load_earth_relief(resolution="01m",region=[12, 34, 31, 41])


# Load the earthquake events data
# Plot PS and BS
ps_data = eve_perf[eve_perf['SR'] != 'BS']
bs_data = eve_perf[eve_perf['SR'] == 'BS']

with fig.subplot(nrows=3,ncols=1, subsize=["16c", "11c"],frame=["ag"],sharex =True): 
    with fig.set_panel(0):
        fig.grdimage(grid=grid, projection=projection, region = region, cmap="geo",transparency=30,shading=True)

        fig.plot(
            x=ps_data['lon'],
            y=ps_data['lat'],
            style="h0.15",
            fill="red",
            label="Subduction Type(PS)",
            projection=projection, 
            region = region, 
        )
        fig.plot(
            x=bs_data['lon'],
            y=bs_data['lat'],
            style="h0.15",
            fill="green",
            label="Crustal Type(BS)",
            projection=projection, 
            region = region, 
        )

        fig.text(
            x=[14, 14.2],
            y=[33, 34],
            text=[f"Events: 53550",f"Locations: 1283"],
            font="15p,Helvetica-Bold,black",
            projection=projection, 
            region = region,
            )

        fig.legend(position="jTR+o1c/1c", box="+gwhite+p1p,black")
        
    with fig.set_panel(1):
        fig.grdimage(grid=grid, projection=projection, region = region, cmap="geo",transparency=30,shading=True)
        cmap = pygmt.makecpt(cmap="rainbow", series=[0, 1, 0.1],continuous =False)
        
        fig.plot(
            x=sample_train['lon'],
            y=sample_train['lat'],
            fill = sample_train['g'],
            style="h0.15",
            projection=projection, 
            region = region,
            cmap = True, )

        fig.text(
            x=[14, 14.2],
            y=[33, 34],
            text=[f"Events: {str(train_count)}", f"Locations: {len(sample_train)}"],
            font="15p,Helvetica-Bold,black",
            projection=projection, 
            region = region,
            )

    with fig.set_panel(2):
        fig.grdimage(grid=grid, projection=projection, region = region, cmap="geo",transparency=30,shading=True)
        cmap = pygmt.makecpt(cmap="rainbow", series=[0, 1, 0.1],continuous =False)
        fig.plot(
            x=sample_bad_bigger['lon'],
            y=sample_bad_bigger['lat'],
            fill = sample_bad_bigger['g'],
            style="h0.15",
            projection=projection, 
            region = region,
            cmap = True, )
        fig.text(
            x=[14, 14.2],
            y=[33, 34],
            text=[f"Events: {str(realbad_count)}",f"Locations: {len(sample_bad_bigger)}"],
            font="15p,Helvetica-Bold,black",
            projection=projection, 
            region = region,
            )
        
        fig.colorbar(frame=["y+lG"])#,position="JMR+o1c/5c+w8c/0.5c+m")

#Finalise and save
fig.savefig(f'./map_db_train_val_{reg}.png',dpi=300)
fig.show()


In [ ]:
import pygmt
import pandas as pd
import matplotlib.pyplot as plt
import contextily as cx
import numpy as np

# Load file
MLDir = '/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami' 
reg = 'SR'
if reg == 'CT':
    columnname = str(38)
    list_size = ['225', '529','892','1658','3454','7071']  
elif reg == 'SR':
    columnname = str(54)
    list_size = ['228', '550' ,'961','1773','3669','6941']

train_size = list_size[3]
eve_perf = pd.read_csv(f'{MLDir}/model/{reg}/multifoldMC/out/model_direct_off[64, 128, 256]_on[16, 128, 128]_{train_size}_compile_combined.csv')
eve_perf = eve_perf.sort_values(by='g', ascending=True)
eve_perf.reset_index(drop=True, inplace=True)

def filter_grids(samplelist):
    empty_table = pd.DataFrame()
    #filter rows per each grid_id, and keep row with lowest g value
    print('count',len(samplelist))
    for grid in samplelist['Location'].unique():
        samplegrid = samplelist[samplelist['Location']==grid]
        samplegrid = samplegrid.sort_values(by='g', ascending=False)
        #remove all rows except first one
        samplegrid = samplegrid.drop(samplegrid.index[1:])
        #append to empty table
        empty_table = pd.concat([empty_table,samplegrid],axis=0)
    return empty_table, len(samplelist)    

sample_train = eve_perf[eve_perf['split']=='train']
sample_train = sample_train[sample_train['max_off']>0.1]
# sample_train = sample_train[sample_train['g']>0.3]
sample_train = sample_train[sample_train['true']>5000]
sample_train,train_count = filter_grids(sample_train)

sample_test = eve_perf[eve_perf['split']=='test']
sample_test = sample_test[sample_test['max_off']>0.1]
sample_test,test_count = filter_grids(sample_test)

sample_bad = eve_perf[eve_perf['split']=='test']
sample_bad = sample_bad[sample_bad['max_off']>0.1]
sample_bad = sample_bad[sample_bad['g']>0.3]
sample_bad,bad_count = filter_grids(sample_bad)
# bad_count = len(sample_bad)

sample_bad_bigger = eve_perf[eve_perf['split']=='test']
sample_bad_bigger = sample_bad_bigger[sample_bad_bigger['max_off']>0.1]
# sample_bad_bigger = sample_bad_bigger[sample_bad_bigger['g']>0.3]
sample_bad_bigger= sample_bad_bigger[sample_bad_bigger['true']>5000]
sample_bad_bigger,realbad_count = filter_grids(sample_bad_bigger)

# Set region and projection
region = [12, 28, 32, 41]  # same as xlim/ylim
projection = "M15c"  # Mercator projection, 15 cm width

# Create two subplot panels
fig = pygmt.Figure()
grid = pygmt.datasets.load_earth_relief(resolution="01m",region=[12, 34, 31, 41])


# Load the earthquake events data
# Plot PS and BS
ps_data = eve_perf[eve_perf['SR'] != 'BS']
bs_data = eve_perf[eve_perf['SR'] == 'BS']

with fig.subplot(nrows=3,ncols=1, subsize=["16c", "11c"],frame=["ag"],sharex =True): 
    with fig.set_panel(0):
        fig.grdimage(grid=grid, projection=projection, region = region, cmap="geo",transparency=30,shading=True)

        fig.plot(
            x=ps_data['lon'],
            y=ps_data['lat'],
            style="h0.15",
            fill="red",
            label="Subduction Type(PS)",
            projection=projection, 
            region = region, 
        )
        fig.plot(
            x=bs_data['lon'],
            y=bs_data['lat'],
            style="h0.15",
            fill="green",
            label="Crustal Type(BS)",
            projection=projection, 
            region = region, 
        )

        fig.text(
            x=[14, 14.2],
            y=[33, 34],
            text=[f"Events: 53550",f"Locations: 1283"],
            font="15p,Helvetica-Bold,black",
            projection=projection, 
            region = region,
            )

        fig.legend(position="jTR+o1c/1c", box="+gwhite+p1p,black")
        
    with fig.set_panel(1):
        fig.grdimage(grid=grid, projection=projection, region = region, cmap="geo",transparency=30,shading=True)
        cmap = pygmt.makecpt(cmap="rainbow", series=[0, 1, 0.1],continuous =False)
        
        fig.plot(
            x=sample_train['lon'],
            y=sample_train['lat'],
            fill = sample_train['g'],
            style="h0.15",
            projection=projection, 
            region = region,
            cmap = True, )

        fig.text(
            x=[14, 14.2],
            y=[33, 34],
            text=[f"Events: {str(train_count)}", f"Locations: {len(sample_train)}"],
            font="15p,Helvetica-Bold,black",
            projection=projection, 
            region = region,
            )

    with fig.set_panel(2):
        fig.grdimage(grid=grid, projection=projection, region = region, cmap="geo",transparency=30,shading=True)
        cmap = pygmt.makecpt(cmap="rainbow", series=[0, 1, 0.1],continuous =False)
        fig.plot(
            x=sample_bad_bigger['lon'],
            y=sample_bad_bigger['lat'],
            fill = sample_bad_bigger['g'],
            style="h0.15",
            projection=projection, 
            region = region,
            cmap = True, )
        fig.text(
            x=[14, 14.2],
            y=[33, 34],
            text=[f"Events: {str(realbad_count)}",f"Locations: {len(sample_bad_bigger)}"],
            font="15p,Helvetica-Bold,black",
            projection=projection, 
            region = region,
            )
        
        fig.colorbar(frame=["y+lG"])

#Finalise and save
fig.savefig(f'./map_db_train_val_{reg}.png',dpi=300)
fig.show()


In [ ]:
sample_train = eve_perf[eve_perf['split']=='train']
sample_train = sample_train[sample_train['max_off']>0.1]
# sample_train = sample_train[sample_train['g']>0.3]
sample_train = sample_train[sample_train['true']>5000]
sample_train,train_count = filter_grids(sample_train)

sample_test = eve_perf[eve_perf['split']=='test']
sample_test = sample_test[sample_test['max_off']>0.1]
sample_test,test_count = filter_grids(sample_test)

sample_bad = eve_perf[eve_perf['split']=='test']
sample_bad = sample_bad[sample_bad['max_off']>0.1]
sample_bad = sample_bad[sample_bad['g']>0.3]
sample_bad,bad_count = filter_grids(sample_bad)
# bad_count = len(sample_bad)

sample_bad_bigger = eve_perf[eve_perf['split']=='test']
sample_bad_bigger = sample_bad_bigger[sample_bad_bigger['max_off']>0.1]
sample_bad_bigger = sample_bad_bigger[sample_bad_bigger['g']>0.3]
sample_bad_bigger= sample_bad_bigger[sample_bad_bigger['true']>5000]
sample_bad_bigger,realbad_count = filter_grids(sample_bad_bigger)

Sample predictions

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable
import pandas as pd
import numpy as np
import matplotlib.colors as mplcolors
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from sklearn.metrics import r2_score
import xarray as xr
from scipy.ndimage import gaussian_filter

#select particular representative gauge
# Load file
MLDir = '/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami' 
SimDir = "/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/data/simu/"
reg = 'CT'
if reg == 'CT':
    columnname = str(38)
    list_size = list_size = ['225', '529','892','1658','3454','7071']  
    control_points =  [[37.5022,15.0960],
            [37.48876,15.08936],
            [37.47193,15.07816],
            [37.46273,15.08527],
            [37.46252,15.08587],
            [37.45312,15.07874],
            [37.42821,15.08506],
            [37.40958,15.08075],
            [37.38595,15.08539],
            [37.35084,15.08575],
            [37.33049,15.07029],
            [37.40675,15.05037]]
    
elif reg == 'SR':
    columnname = str(54)
    list_size = ['228', '550' ,'961','1773','3669','6941']
    control_points = [[37.01,15.29],
            [37.06757,15.28709],
            [37.05266,15.26536],
            [37.03211,15.28632]]
mask_size = list_size[2]
train_size = list_size[3]

#predictions and post processed predictions
idx = np.load(f'/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/data/processed/lat_lon_idx_{reg}_{mask_size}.npy')
pred_depths = np.load(f'/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/model/{reg}/multifoldMC/PTHA/pred_d_{train_size}_direct.npy', mmap_mode='r')
true_depths = np.load(f'/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/model/{reg}/multifoldMC/PTHA/true_d_53550.npy', mmap_mode='r')
eve_perf = pd.read_csv(f'/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/model/{reg}/multifoldMC/out/model_direct_off[64, 128, 256]_on[16, 128, 128]_{train_size}_compile_combined.csv')    

In [ ]:
ids = [
        'BS_manning003/E01267N3753E01646N3535-BS-M809_E01502N3737_D144_S022D70R270_A006995_S075',
        'PS_manning003/E02020N3739E02658N3366-PS-Str_PYes_Var-M902_E02417N3454_S001',
        'BS_manning003/E01267N3753E01646N3535-BS-M809_E01547N3670_D010_S337D70R270_A006995_S075',
        'BS_manning003/E01267N3753E01646N3535-BS-M809_E01495N3692_D010_S022D50R270_A006995_S075',
        'BS_4-8_manning003/E01267N3753E01646N3535-BS-M809_E01502N3737_D010_S067D90R090_A006995_S075'
        ]
for id in ids:
    eve_id = np.loadtxt('/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/data/events/sample_events53550.txt',dtype='str')
    eve = np.where(eve_id==id)[0][0]
    print(id,'\n',eve)
    # eve=32145
    # id =eve_id[eve]

    #read dZ file and grid location file to extract location information
    data2plot = xr.open_dataset(f'/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/data/simu/{id}/{reg}_deformation.nc')
    dz = data2plot['deformation'].values
    # (948 x 1300)
    # x = data2plot['x'].values #1300
    # y = data2plot['y'].values #948

    x = np.linspace(0,dz.shape[1],dz.shape[1])
    y = np.linspace(0,dz.shape[0],dz.shape[0])

    #create list of x,y,dz
    xy_mesh = np.meshgrid(x,y)
    # dz_smooth = gaussian_filter(dz, sigma=50)
    dz_smooth = dz
    x_list,y_list = xy_mesh[0].flatten(),xy_mesh[1].flatten()
    dz_list = dz.flatten()

    pred=pred_depths[eve]
    true=true_depths[eve]

    fig, axs = plt.subplots(1, 4, figsize=(10,8))
    axs = axs.ravel()

    # Plot performance variable values
    cmap_depth = plt.get_cmap('twilight',20)
    cmap_error = plt.get_cmap('seismic', 10)
    cmap_dz = plt.get_cmap('RdYlGn_r',10)

    # Set NaN for rows where count_test is less than 1
    true=true/100
    pred=pred/100
    pred= np.where(pred < 0.1, np.nan, pred)
    true= np.where(true < 0.1, np.nan, true)
    error1=true-pred
    error2=pred-true
    error = np.where(np.abs(error1) < np.abs(error2), error1, -error2)
    error= np.where((error < 0.01) & (error > -0.01), np.nan, error)

    depth_max = max(np.nanmax(pred),np.nanmax(true))
    error_max = round(np.nanmax(np.abs(error)))
    dz_max = round(np.nanmax(np.abs(dz_smooth)))


    # Local Deformation
    DZ = axs[0].scatter(x_list,y_list, c=dz_smooth, s=0.0005, cmap=cmap_dz,
                        vmin=-5, vmax=5,alpha=1)
    axs[0].text(0.2, 0.95, f'max: {np.nanmax(dz_smooth):.2f},\nmin: {np.nanmin(dz_smooth):.2f}',
                horizontalalignment='center', verticalalignment='center',   
                transform=axs[0].transAxes, fontsize=12)
    axs[0].set_title('Local Deformation')

    # True
    TR = axs[1].scatter(idx[:, 1], idx[:, 0], c=true, s=0.0005, cmap=cmap_depth,
                         vmin=0, vmax=10,alpha=1)
    axs[1].text(0.2, 0.95, f'max: {np.nanmax(true):.2f}', 
                horizontalalignment='center', verticalalignment='center',
                transform=axs[1].transAxes, fontsize=12)
    axs[1].set_title('True')

    # Pred
    PR = axs[2].scatter(idx[:, 1], idx[:, 0], c=pred, s=0.0005, cmap=cmap_depth,
                         vmin=0, vmax=10,alpha=1)
    axs[2].text(0.2, 0.95, f'max: {np.nanmax(pred):.2f}\nr^2: {eve_perf["r2"].iloc[eve]:.2f}\ng: {eve_perf["g"].iloc[eve]:.2f}',
                horizontalalignment='center', verticalalignment='center',   
                transform=axs[2].transAxes, fontsize=12)
    axs[2].set_title('Prediction')

    # Error
    ER = axs[3].scatter(idx[:, 1], idx[:, 0], c=error, s=0.0005, cmap=cmap_error,
                        vmin=-5,vmax=5,alpha=1)
    axs[3].text(0.2, 0.95, f'max: {np.nanmax(error):.2f},\nmin: {np.nanmin(error):.2f}',
                horizontalalignment='center', verticalalignment='center',
                transform=axs[3].transAxes, fontsize=12)
    axs[3].set_title('Error')


    # Set axis scale as equal and add gridlines
    for ax in axs:
        #keep gridlines but turn off axis borders and ticks
        ax.set_aspect('equal')
        ax.set_axis_off()
        ax.set_xlim([0, max(idx[:, 1])])
        ax.set_ylim([0, max(idx[:, 0])])
        ax.hlines(y=np.arange(0, max(idx[:, 0]), 150), xmin=0, xmax=max(idx[:, 1]), color='grey', linestyle='--', linewidth=0.5,alpha=0.75)
        ax.vlines(x=np.arange(0, max(idx[:, 1]), 150), ymin=0, ymax=max(idx[:, 0]), color='grey', linestyle='--', linewidth=0.5,alpha=0.75)

    # Add a common colorbar for the whole fig using axes transform
    cbar_dz = fig.add_axes([0.02, 0.1, 0.22, 0.02])
    cbar_dep = fig.add_axes([0.28, 0.1, 0.46, 0.02])
    cbar_err = fig.add_axes([0.78, 0.1, 0.22, 0.02])

    cbar1 = fig.colorbar(TR, cax=cbar_dep, orientation ='horizontal',extend='max')
    cbar2 = fig.colorbar(ER, cax=cbar_err, orientation ='horizontal',extend='both')
    cbar3 = fig.colorbar(DZ, cax=cbar_dz, orientation ='horizontal',extend='both')
    cbar1.ax.tick_params(labelsize=12)
    cbar2.ax.tick_params(labelsize=12)
    cbar3.ax.tick_params(labelsize=12)
    cbar1.set_label('Depth(m)', fontsize=12)
    cbar2.set_label('Error(m)', fontsize=12)
    cbar3.set_label('Local Deform.(m)', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'./sample_TPE_{train_size}_{reg}_{str(eve)}.png', dpi=300, bbox_inches='tight', pad_inches=0.1)

In [ ]:
ids = [
        'BS_manning003/E01267N3753E01646N3535-BS-M809_E01523N3692_D010_S292D50R270_A006995_S075',
        'PS_manning003/E02020N3739E02658N3366-PS-Str_PYes_Var-M895_E02351N3465_S003',
        'BS_manning003/E01267N3753E01646N3535-BS-M809_E01547N3670_D010_S337D70R270_A006995_S075',
        'BS_4-8_manning003/E01267N3753E01646N3535-BS-M809_E01551N3692_D010_S112D90R090_A006995_S075',
        ]
for id in ids:
    eve_id = np.loadtxt('/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/data/events/sample_events53550.txt',dtype='str')
    eve = np.where(eve_id==id)[0][0]
    print(id,'\n',eve)
    # eve=32145
    # id =eve_id[eve]

    #read dZ file and grid location file to extract location information
    data2plot = xr.open_dataset(f'/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/data/simu/{id}/{reg}_deformation.nc')
    dz = data2plot['deformation'].values
    # (948 x 1300)
    # x = data2plot['x'].values #1300
    # y = data2plot['y'].values #948

    x = np.linspace(0,dz.shape[1],dz.shape[1])
    y = np.linspace(0,dz.shape[0],dz.shape[0])

    #create list of x,y,dz
    xy_mesh = np.meshgrid(x,y)
    # dz_smooth = gaussian_filter(dz, sigma=50)
    dz_smooth = dz
    x_list,y_list = xy_mesh[0].flatten(),xy_mesh[1].flatten()
    dz_list = dz.flatten()

    pred=pred_depths[eve]
    true=true_depths[eve]

    fig, axs = plt.subplots(1, 4, figsize=(10,3))
    axs = axs.ravel()

    # Plot performance variable values
    cmap_depth = plt.get_cmap('twilight',20)
    cmap_error = plt.get_cmap('seismic', 10)
    cmap_dz = plt.get_cmap('RdYlGn_r',10)

    # Set NaN for rows where count_test is less than 1
    true=true/100
    pred=pred/100
    pred= np.where(pred < 0.1, np.nan, pred)
    true= np.where(true < 0.1, np.nan, true)
    error1=true-pred
    error2=pred-true
    error = np.where(np.abs(error1) < np.abs(error2), error1, -error2)
    error= np.where((error < 0.01) & (error > -0.01), np.nan, error)

    depth_max = max(np.nanmax(pred),np.nanmax(true))

    error_max = round(np.nanmax(np.abs(error)))
    dz_max = round(np.nanmax(np.abs(dz_smooth)))

    # Local Deformation
    DZ = axs[0].scatter(x_list,y_list, c=dz_smooth, s=0.0001, cmap=cmap_dz,
                        vmin=-5, vmax=5,alpha=1)
    axs[0].text(0.25, 0.25, f'max: {np.nanmax(dz_smooth):.2f},\nmin: {np.nanmin(dz_smooth):.2f}',
                horizontalalignment='center', verticalalignment='center',
                transform=axs[0].transAxes, fontsize=12)
    axs[0].set_title('Local Deformation')

    # True
    TR = axs[1].scatter(idx[:, 1], idx[:, 0], c=true, s=0.0001, cmap=cmap_depth, 
                        vmin=0, vmax=depth_max,alpha=1)
    axs[1].text(0.25, 0.25, f'max: {np.nanmax(true):.2f}',
                horizontalalignment='center', verticalalignment='center',
                transform=axs[1].transAxes, fontsize=12)
    axs[1].set_title('True')

    # Pred
    PR = axs[2].scatter(idx[:, 1], idx[:, 0], c=pred, s=0.0001, cmap=cmap_depth,
                         vmin=0, vmax=depth_max,alpha=1)
    axs[2].text(0.25, 0.25, f'max: {np.nanmax(pred):.2f}\nr^2: {eve_perf["r2"].iloc[eve]:.2f}\ng: {eve_perf["g"].iloc[eve]:.2f}',
                horizontalalignment='center', verticalalignment='center',
                transform=axs[2].transAxes, fontsize=12)
    axs[2].set_title('Prediction')

    # Error
    ER = axs[3].scatter(idx[:, 1], idx[:, 0], c=error, s=0.0001, cmap=cmap_error,
                        vmin=-5,vmax=5,alpha=1)
    axs[3].text(0.25, 0.25, f'max: {np.nanmax(error):.2f},\nmin: {np.nanmin(error):.2f}',
                horizontalalignment='center', verticalalignment='center',
                transform=axs[3].transAxes, fontsize=12)
    axs[3].set_title('Error')

    # Set axis scale as equal and add gridlines
    for ax in axs:
        #keep gridlines but turn off axis borders and ticks
        ax.set_aspect('equal')
        ax.set_axis_off()
        ax.set_xlim([0, max(idx[:, 1])])
        ax.set_ylim([0, max(idx[:, 0])])
        ax.hlines(y=np.arange(0, max(idx[:, 0]), 150), xmin=0, xmax=max(idx[:, 1]), color='grey', linestyle='--', linewidth=0.5,alpha=0.75)
        ax.vlines(x=np.arange(0, max(idx[:, 1]), 150), ymin=0, ymax=max(idx[:, 0]), color='grey', linestyle='--', linewidth=0.5,alpha=0.75)

    # Add a common colorbar for the whole fig using axes transform
    cbar_dz = fig.add_axes([0.02, 0.15, 0.22, 0.02])
    cbar_dep = fig.add_axes([0.28, 0.15, 0.46, 0.02])
    cbar_err = fig.add_axes([0.78, 0.15, 0.22, 0.02])

    cbar1 = fig.colorbar(TR, cax=cbar_dep, orientation ='horizontal',extend='max')
    cbar2 = fig.colorbar(ER, cax=cbar_err, orientation ='horizontal',extend='both')
    cbar3 = fig.colorbar(DZ, cax=cbar_dz, orientation ='horizontal',extend='both')
    cbar1.ax.tick_params(labelsize=12)
    cbar2.ax.tick_params(labelsize=12)
    cbar3.ax.tick_params(labelsize=12)
    cbar1.set_label('Depth(m)', fontsize=12)
    cbar2.set_label('Error(m)', fontsize=12)
    cbar3.set_label('Local Deform.(m)', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'./sample_TPE_{train_size}_{reg}_{str(eve)}_nodeform.png', dpi=300, bbox_inches='tight', pad_inches=0.1)